# Week 2 Assignment – SQL Data Analysis

## Step 0: Setup – Install & Import Libraries

In [1]:
import pandas as pd
import sqlite3
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

Libraries loaded successfully!


## Step 1: Load Dataset into SQLite Database

In [2]:
# Load CSV into pandas
df = pd.read_csv('Sample - Superstore.csv', encoding='latin-1')

# Clean column names: remove spaces and special chars
df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace('-', '_')

# Parse Order_Date and Ship_Date
df['Order_Date'] = pd.to_datetime(df['Order_Date'], dayfirst=False)
df['Ship_Date']  = pd.to_datetime(df['Ship_Date'],  dayfirst=False)

# Create in-memory SQLite DB and load table
conn = sqlite3.connect(':memory:')
df.to_sql('superstore', conn, if_exists='replace', index=False)

print(f'Dataset loaded! Shape: {df.shape}')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')

Dataset loaded! Shape: (9994, 21)
Rows: 9,994 | Columns: 21


---
## Step 2: Explore the Table – Schema & Sample Data

In [3]:
# 2.1 View all column names and data types
print('=== TABLE SCHEMA ===')
schema = pd.read_sql("PRAGMA table_info(superstore)", conn)
print(schema[['name', 'type']].to_string(index=False))

=== TABLE SCHEMA ===
         name      type
       Row_ID   INTEGER
     Order_ID      TEXT
   Order_Date TIMESTAMP
    Ship_Date TIMESTAMP
    Ship_Mode      TEXT
  Customer_ID      TEXT
Customer_Name      TEXT
      Segment      TEXT
      Country      TEXT
         City      TEXT
        State      TEXT
  Postal_Code   INTEGER
       Region      TEXT
   Product_ID      TEXT
     Category      TEXT
 Sub_Category      TEXT
 Product_Name      TEXT
        Sales      REAL
     Quantity   INTEGER
     Discount      REAL
       Profit      REAL


In [4]:
# 2.2 Sample 5 rows
print('=== SAMPLE DATA (5 rows) ===')
sample = pd.read_sql("SELECT * FROM superstore LIMIT 5", conn)
sample

=== SAMPLE DATA (5 rows) ===


,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08 00:00:00,2016-11-11 00:00:00,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08 00:00:00,2016-11-11 00:00:00,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12 00:00:00,2016-06-16 00:00:00,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11 00:00:00,2015-10-18 00:00:00,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11 00:00:00,2015-10-18 00:00:00,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [5]:
# 2.3 Total row count
count = pd.read_sql("SELECT COUNT(*) AS total_rows FROM superstore", conn)
print('Total rows:', count['total_rows'][0])

Total rows: 9994


In [6]:
# 2.4 Distinct values in key categorical columns
distinct = pd.read_sql("""
    SELECT 
        COUNT(DISTINCT Region)   AS distinct_regions,
        COUNT(DISTINCT Category) AS distinct_categories,
        COUNT(DISTINCT Segment)  AS distinct_segments,
        COUNT(DISTINCT State)    AS distinct_states
    FROM superstore
""", conn)
distinct

,distinct_regions,distinct_categories,distinct_segments,distinct_states
0,4,3,3,49


---
## Step 3: WHERE Filters – Region, Category, Date, Sales

In [7]:
# 3.1 Filter by Region = 'West'
print('=== Orders from West Region ===')
west = pd.read_sql("""
    SELECT Order_ID, Order_Date, Customer_Name, Category, Sales, Profit
    FROM superstore
    WHERE Region = 'West'
    LIMIT 10
""", conn)
west

=== Orders from West Region ===


,Order_ID,Order_Date,Customer_Name,Category,Sales,Profit
0,CA-2016-138688,2016-06-12 00:00:00,Darrin Van Huff,Office Supplies,14.620,6.8714
1,CA-2014-115812,2014-06-09 00:00:00,Brosina Hoffman,Furniture,48.860,14.1694
2,CA-2014-115812,2014-06-09 00:00:00,Brosina Hoffman,Office Supplies,7.280,1.9656
3,CA-2014-115812,2014-06-09 00:00:00,Brosina Hoffman,Technology,907.152,90.7152
4,CA-2014-115812,2014-06-09 00:00:00,Brosina Hoffman,Office Supplies,18.504,5.7825
5,CA-2014-115812,2014-06-09 00:00:00,Brosina Hoffman,Office Supplies,114.900,34.4700
6,CA-2014-115812,2014-06-09 00:00:00,Brosina Hoffman,Furniture,1706.184,85.3092
7,CA-2014-115812,2014-06-09 00:00:00,Brosina Hoffman,Technology,911.424,68.3568
8,CA-2016-161389,2016-12-05 00:00:00,Irene Maddox,Office Supplies,407.976,132.5922
9,CA-2014-167164,2014-05-13 00:00:00,Alejandro Grove,Office Supplies,55.500,9.9900


In [8]:
# 3.2 Filter by Category = 'Technology'
print('=== Technology Category Orders ===')
tech = pd.read_sql("""
    SELECT Order_ID, Product_Name, Sales, Quantity, Profit
    FROM superstore
    WHERE Category = 'Technology'
    LIMIT 10
""", conn)
tech

=== Technology Category Orders ===


,Order_ID,Product_Name,Sales,Quantity,Profit
0,CA-2014-115812,Mitel 5320 IP Phone VoIP phone,907.152,6,90.7152
1,CA-2014-115812,Konftel 250 Conference phone - Charcoal black,911.424,4,68.3568
2,CA-2014-143336,Cisco SPA 501G IP Phone,213.480,3,16.0110
3,CA-2016-121755,Imation 8GB Mini TravelDrive USB 2.0 Flash Drive,90.570,3,11.7741
4,CA-2016-117590,GE 30524EE4,1097.544,7,123.4737
5,CA-2015-117415,Plantronics HL10 Handset Lifter,371.168,4,41.7564
6,CA-2017-120999,Panasonic Kx-TS550,147.168,4,16.5564
7,CA-2016-118255,Verbatim 25 GB 6x Blu-ray Single Layer Recorda...,45.980,2,19.7714
8,CA-2016-169194,Imation 8gb Micro Traveldrive Usb 2.0 Flash Drive,45.000,3,4.9500
9,CA-2016-169194,"LF Elite 3D Dazzle Designer Hard Case Cover, L...",21.800,2,6.1040


In [9]:
# 3.3 Filter by Date Range (2017)
print('=== Orders in Year 2017 ===')
orders_2017 = pd.read_sql("""
    SELECT Order_ID, Order_Date, Customer_Name, Sales
    FROM superstore
    WHERE Order_Date BETWEEN '2017-01-01' AND '2017-12-31'
    LIMIT 10
""", conn)
orders_2017

=== Orders in Year 2017 ===


,Order_ID,Order_Date,Customer_Name,Sales
0,CA-2017-114412,2017-04-15 00:00:00,Andrew Allen,15.552
1,US-2017-156909,2017-07-16 00:00:00,Sandra Flanagan,71.372
2,CA-2017-107727,2017-10-19 00:00:00,Matt Abelman,29.472
3,CA-2017-120999,2017-09-10 00:00:00,Linda Cazamias,147.168
4,CA-2017-139619,2017-09-19 00:00:00,Erin Smith,95.616
5,CA-2017-114440,2017-09-14 00:00:00,Tracy Blumstein,19.050
6,US-2017-118038,2017-12-09 00:00:00,Ken Brennan,1.248
7,US-2017-118038,2017-12-09 00:00:00,Ken Brennan,9.708
8,US-2017-118038,2017-12-09 00:00:00,Ken Brennan,27.240
9,US-2017-119662,2017-11-13 00:00:00,Christopher Schild,230.376


In [10]:
# 3.4 Filter by high-value sales (Sales > 1000)
print('=== High-Value Orders (Sales > $1000) ===')
high_sales = pd.read_sql("""
    SELECT Order_ID, Customer_Name, Product_Name, Sales, Profit, Region
    FROM superstore
    WHERE Sales > 1000
    ORDER BY Sales DESC
    LIMIT 10
""", conn)
high_sales

=== High-Value Orders (Sales > $1000) ===


,Order_ID,Customer_Name,Product_Name,Sales,Profit,Region
0,CA-2014-145317,Sean Miller,Cisco TelePresence System EX90 Videoconferenci...,22638.480,-1811.0784,South
1,CA-2016-118689,Tamara Chand,Canon imageCLASS 2200 Advanced Copier,17499.950,8399.9760,Central
2,CA-2017-140151,Raymond Buch,Canon imageCLASS 2200 Advanced Copier,13999.960,6719.9808,West
3,CA-2017-127180,Tom Ashbrook,Canon imageCLASS 2200 Advanced Copier,11199.968,3919.9888,East
4,CA-2017-166709,Hunter Lopez,Canon imageCLASS 2200 Advanced Copier,10499.970,5039.9856,East
5,CA-2016-117121,Adrian Barton,GBC Ibimaster 500 Manual ProClick Binding System,9892.740,4946.3700,Central
6,CA-2014-116904,Sanjit Chand,Ibico EPK-21 Electric Binding System,9449.950,4630.4755,Central
7,US-2016-107440,Bill Shonely,"3D Systems Cube Printer, 2nd Generation, Magenta",9099.930,2365.9818,East
8,CA-2016-158841,Sanjit Engle,HP Designjet T520 Inkjet Large Format Printer ...,8749.950,2799.9840,South
9,CA-2016-143714,Christopher Conant,Canon imageCLASS 2200 Advanced Copier,8399.976,1119.9968,East


In [11]:
# 3.5 Combined Filter: West + Technology + profitable orders
print('=== West Region | Technology | Profit > 0 ===')
combined = pd.read_sql("""
    SELECT Order_ID, Customer_Name, Product_Name, Sales, Profit
    FROM superstore
    WHERE Region = 'West'
      AND Category = 'Technology'
      AND Profit > 0
    ORDER BY Profit DESC
    LIMIT 10
""", conn)
combined

=== West Region | Technology | Profit > 0 ===


,Order_ID,Customer_Name,Product_Name,Sales,Profit
0,CA-2017-140151,Raymond Buch,Canon imageCLASS 2200 Advanced Copier,13999.960,6719.9808
1,CA-2015-111829,Fred Hopkins,Canon PC940 Copier,3149.930,1480.4671
2,CA-2017-127432,Alan Dominguez,Canon Image Class D660 Copier,2999.950,1379.9770
3,CA-2017-133865,Penelope Sewall,Canon Imageclass D680 Copier / Fax,3359.952,1049.9850
4,CA-2017-145219,Robert Marley,Hewlett Packard LaserJet 3310 Copier,2879.952,1007.9832
5,CA-2016-135265,Christopher Conant,Canon PC1060 Personal Laser Copier,2799.960,944.9865
6,CA-2016-129630,Ionia McGrath,Canon PC1060 Personal Laser Copier,2799.960,944.9865
7,US-2017-135013,Harold Ryan,Hewlett Packard LaserJet 3310 Copier,2399.960,839.9860
8,CA-2017-168109,Jim Kriz,Plantronics Savi W720 Multi-Device Wireless He...,1687.800,742.6320
9,US-2014-135972,Jack Garza,Canon PC1080F Personal Copier,1799.970,701.9883


---
## Step 4: GROUP BY – Aggregations (Sales, Quantity, Averages)

In [12]:
# 4.1 Total Sales & Profit by Region
print('=== Sales & Profit by Region ===')
by_region = pd.read_sql("""
    SELECT 
        Region,
        COUNT(*) AS total_orders,
        ROUND(SUM(Sales), 2)   AS total_sales,
        ROUND(SUM(Profit), 2)  AS total_profit,
        ROUND(AVG(Sales), 2)   AS avg_sales
    FROM superstore
    GROUP BY Region
    ORDER BY total_sales DESC
""", conn)
by_region

=== Sales & Profit by Region ===


,Region,total_orders,total_sales,total_profit,avg_sales
0,West,3203,725457.82,108418.45,226.49
1,East,2848,678781.24,91522.78,238.34
2,Central,2323,501239.89,39706.36,215.77
3,South,1620,391721.91,46749.43,241.80


In [13]:
# 4.2 Total Sales by Category
print('=== Sales by Category ===')
by_cat = pd.read_sql("""
    SELECT 
        Category,
        COUNT(*) AS total_orders,
        ROUND(SUM(Sales), 2)      AS total_sales,
        ROUND(SUM(Quantity), 0)   AS total_quantity,
        ROUND(AVG(Discount), 4)   AS avg_discount
    FROM superstore
    GROUP BY Category
    ORDER BY total_sales DESC
""", conn)
by_cat

=== Sales by Category ===


,Category,total_orders,total_sales,total_quantity,avg_discount
0,Technology,1847,836154.03,6939.0,0.1323
1,Furniture,2121,741999.80,8028.0,0.1739
2,Office Supplies,6026,719047.03,22906.0,0.1573


In [14]:
# 4.3 Sales by Sub-Category
print('=== Sales by Sub-Category ===')
by_subcat = pd.read_sql("""
    SELECT 
        Sub_Category,
        ROUND(SUM(Sales), 2)  AS total_sales,
        ROUND(SUM(Profit), 2) AS total_profit,
        SUM(Quantity)         AS total_quantity
    FROM superstore
    GROUP BY Sub_Category
    ORDER BY total_sales DESC
""", conn)
by_subcat

=== Sales by Sub-Category ===


,Sub_Category,total_sales,total_profit,total_quantity
0,Phones,330007.05,44515.73,3289
1,Chairs,328449.10,26590.17,2356
2,Storage,223843.61,21278.83,3158
3,Tables,206965.53,-17725.48,1241
4,Binders,203412.73,30221.76,5974
5,Machines,189238.63,3384.76,440
6,Accessories,167380.32,41936.64,2976
7,Copiers,149528.03,55617.82,234
8,Bookcases,114880.00,-3472.56,868
9,Appliances,107532.16,18138.01,1729


In [15]:
# 4.4 Sales by Segment
print('=== Sales by Customer Segment ===')
by_segment = pd.read_sql("""
    SELECT 
        Segment,
        COUNT(DISTINCT Customer_ID) AS unique_customers,
        ROUND(SUM(Sales), 2)        AS total_sales,
        ROUND(AVG(Sales), 2)        AS avg_order_value
    FROM superstore
    GROUP BY Segment
    ORDER BY total_sales DESC
""", conn)
by_segment

=== Sales by Customer Segment ===


,Segment,unique_customers,total_sales,avg_order_value
0,Consumer,409,1161401.34,223.73
1,Corporate,236,706146.37,233.82
2,Home Office,148,429653.15,240.97


---
## Step 5: ORDER BY & LIMIT – Top Products & Categories

In [16]:
# 5.1 Top 10 Products by Sales
print('=== Top 10 Products by Sales ===')
top_products = pd.read_sql("""
    SELECT 
        Product_Name,
        Category,
        ROUND(SUM(Sales), 2)  AS total_sales,
        ROUND(SUM(Profit), 2) AS total_profit
    FROM superstore
    GROUP BY Product_Name, Category
    ORDER BY total_sales DESC
    LIMIT 10
""", conn)
top_products

=== Top 10 Products by Sales ===


,Product_Name,Category,total_sales,total_profit
0,Canon imageCLASS 2200 Advanced Copier,Technology,61599.82,25199.93
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,27453.38,7753.04
2,Cisco TelePresence System EX90 Videoconferenci...,Technology,22638.48,-1811.08
3,HON 5400 Series Task Chairs for Big and Tall,Furniture,21870.58,0.00
4,GBC DocuBind TL300 Electric Binding System,Office Supplies,19823.48,2233.51
5,GBC Ibimaster 500 Manual ProClick Binding System,Office Supplies,19024.50,760.98
6,Hewlett Packard LaserJet 3310 Copier,Technology,18839.69,6983.88
7,HP Designjet T520 Inkjet Large Format Printer ...,Technology,18374.90,4094.98
8,GBC DocuBind P400 Electric Binding System,Office Supplies,17965.07,-1878.17
9,High Speed Automatic Electric Letter Opener,Office Supplies,17030.31,-262.00


In [17]:
# 5.2 Top 10 Most Profitable Products
print('=== Top 10 Most Profitable Products ===')
top_profit = pd.read_sql("""
    SELECT 
        Product_Name,
        Category,
        ROUND(SUM(Profit), 2) AS total_profit,
        ROUND(SUM(Sales), 2)  AS total_sales
    FROM superstore
    GROUP BY Product_Name, Category
    ORDER BY total_profit DESC
    LIMIT 10
""", conn)
top_profit

=== Top 10 Most Profitable Products ===


,Product_Name,Category,total_profit,total_sales
0,Canon imageCLASS 2200 Advanced Copier,Technology,25199.93,61599.82
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,7753.04,27453.38
2,Hewlett Packard LaserJet 3310 Copier,Technology,6983.88,18839.69
3,Canon PC1060 Personal Laser Copier,Technology,4570.93,11619.83
4,HP Designjet T520 Inkjet Large Format Printer ...,Technology,4094.98,18374.90
5,Ativa V4110MDD Micro-Cut Shredder,Technology,3772.95,7699.89
6,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,3717.97,14299.89
7,Plantronics Savi W720 Multi-Device Wireless He...,Technology,3696.28,9367.29
8,Ibico EPK-21 Electric Binding System,Office Supplies,3345.28,15875.92
9,Zebra ZM400 Thermal Label Printer,Technology,3343.54,6965.70


In [18]:
# 5.3 Bottom 5 Loss-Making Products
print('=== Top 5 Loss-Making Products ===')
loss_products = pd.read_sql("""
    SELECT 
        Product_Name,
        Category,
        ROUND(SUM(Profit), 2) AS total_profit
    FROM superstore
    GROUP BY Product_Name, Category
    ORDER BY total_profit ASC
    LIMIT 5
""", conn)
loss_products

=== Top 5 Loss-Making Products ===


,Product_Name,Category,total_profit
0,Cubify CubeX 3D Printer Double Head Print,Technology,-8879.97
1,Lexmark MX611dhe Monochrome Laser Printer,Technology,-4589.97
2,Cubify CubeX 3D Printer Triple Head Print,Technology,-3839.99
3,Chromcraft Bull-Nose Wood Oval Conference Tabl...,Furniture,-2876.12
4,Bush Advantage Collection Racetrack Conference...,Furniture,-1934.40


In [19]:
# 5.4 Top 5 States by Sales
print('=== Top 5 States by Sales ===')
top_states = pd.read_sql("""
    SELECT 
        State,
        ROUND(SUM(Sales), 2) AS total_sales
    FROM superstore
    GROUP BY State
    ORDER BY total_sales DESC
    LIMIT 5
""", conn)
top_states

=== Top 5 States by Sales ===


,State,total_sales
0,California,457687.63
1,New York,310876.27
2,Texas,170188.05
3,Washington,138641.27
4,Pennsylvania,116511.91


---
## Step 6: Business Use Cases

In [20]:
# 6.1 Monthly Sales Trend (all years)
print('=== Monthly Sales Trend ===')
monthly = pd.read_sql("""
    SELECT 
        strftime('%Y', Order_Date) AS year,
        strftime('%m', Order_Date) AS month,
        ROUND(SUM(Sales), 2)       AS monthly_sales,
        ROUND(SUM(Profit), 2)      AS monthly_profit,
        COUNT(*)                   AS total_orders
    FROM superstore
    GROUP BY year, month
    ORDER BY year, month
""", conn)
monthly

=== Monthly Sales Trend ===


,year,month,monthly_sales,monthly_profit,total_orders
0,2014,01,14236.90,2450.19,79
1,2014,02,4519.89,862.31,46
2,2014,03,55691.01,498.73,157
3,2014,04,28295.35,3488.84,135
4,2014,05,23648.29,2738.71,122
5,2014,06,34595.13,4976.52,135
6,2014,07,33946.39,-841.48,143
7,2014,08,27909.47,5318.10,153
8,2014,09,81777.35,8328.10,268
9,2014,10,31453.39,3448.26,159


In [21]:
# 6.2 Top 10 Customers by Sales
print('=== Top 10 Customers by Total Sales ===')
top_customers = pd.read_sql("""
    SELECT 
        Customer_ID,
        Customer_Name,
        Segment,
        ROUND(SUM(Sales), 2)  AS total_sales,
        ROUND(SUM(Profit), 2) AS total_profit,
        COUNT(*)              AS total_orders
    FROM superstore
    GROUP BY Customer_ID, Customer_Name, Segment
    ORDER BY total_sales DESC
    LIMIT 10
""", conn)
top_customers

=== Top 10 Customers by Total Sales ===


,Customer_ID,Customer_Name,Segment,total_sales,total_profit,total_orders
0,SM-20320,Sean Miller,Home Office,25043.05,-1980.74,15
1,TC-20980,Tamara Chand,Corporate,19052.22,8981.32,12
2,RB-19360,Raymond Buch,Consumer,15117.34,6976.10,18
3,TA-21385,Tom Ashbrook,Home Office,14595.62,4703.79,10
4,AB-10105,Adrian Barton,Consumer,14473.57,5444.81,20
5,KL-16645,Ken Lonsdale,Consumer,14175.23,806.85,29
6,SC-20095,Sanjit Chand,Consumer,14142.33,5757.41,22
7,HL-15040,Hunter Lopez,Consumer,12873.30,5622.43,11
8,SE-20110,Sanjit Engle,Consumer,12209.44,2650.68,19
9,CC-12370,Christopher Conant,Consumer,12129.07,2177.05,11


In [22]:
# 6.3 Yearly Sales Growth
print('=== Year-wise Sales Summary ===')
yearly = pd.read_sql("""
    SELECT 
        strftime('%Y', Order_Date) AS year,
        COUNT(DISTINCT Order_ID)   AS unique_orders,
        ROUND(SUM(Sales), 2)       AS total_sales,
        ROUND(SUM(Profit), 2)      AS total_profit,
        ROUND(AVG(Sales), 2)       AS avg_order_value
    FROM superstore
    GROUP BY year
    ORDER BY year
""", conn)
yearly

=== Year-wise Sales Summary ===


,year,unique_orders,total_sales,total_profit,avg_order_value
0,2014,969,484247.50,49543.97,242.97
1,2015,1038,470532.51,61618.60,223.85
2,2016,1315,609205.60,81795.17,235.49
3,2017,1687,733215.26,93439.27,221.38


In [23]:
# 6.4 Ship Mode Performance
print('=== Orders & Sales by Ship Mode ===')
ship_mode = pd.read_sql("""
    SELECT 
        Ship_Mode,
        COUNT(*) AS total_orders,
        ROUND(SUM(Sales), 2)  AS total_sales,
        ROUND(AVG(Sales), 2)  AS avg_sales
    FROM superstore
    GROUP BY Ship_Mode
    ORDER BY total_orders DESC
""", conn)
ship_mode

=== Orders & Sales by Ship Mode ===


,Ship_Mode,total_orders,total_sales,avg_sales
0,Standard Class,5968,1358215.74,227.58
1,Second Class,1945,459193.57,236.09
2,First Class,1538,351428.42,228.50
3,Same Day,543,128363.13,236.40


In [24]:
# 6.5 Discount Impact on Profit
print('=== Discount Brackets vs Avg Profit ===')
discount_impact = pd.read_sql("""
    SELECT 
        CASE 
            WHEN Discount = 0        THEN 'No Discount'
            WHEN Discount <= 0.10    THEN '1-10%'
            WHEN Discount <= 0.20    THEN '11-20%'
            WHEN Discount <= 0.40    THEN '21-40%'
            ELSE 'Above 40%'
        END AS discount_bracket,
        COUNT(*)              AS total_orders,
        ROUND(AVG(Profit), 2) AS avg_profit,
        ROUND(SUM(Sales), 2)  AS total_sales
    FROM superstore
    GROUP BY discount_bracket
    ORDER BY avg_profit DESC
""", conn)
discount_impact

=== Discount Brackets vs Avg Profit ===


,discount_bracket,total_orders,avg_profit,total_sales
0,1-10%,94,96.06,54369.35
1,No Discount,4798,66.90,1087908.47
2,11-20%,3709,24.74,792152.89
3,21-40%,460,-77.86,234137.90
4,Above 40%,933,-106.71,128632.25


In [25]:
# 6.6 Detect Duplicate Order IDs (same Order_ID + Product)
print('=== Checking for Duplicate Order + Product Combinations ===')
duplicates = pd.read_sql("""
    SELECT 
        Order_ID,
        Product_ID,
        COUNT(*) AS occurrence_count
    FROM superstore
    GROUP BY Order_ID, Product_ID
    HAVING COUNT(*) > 1
    ORDER BY occurrence_count DESC
    LIMIT 10
""", conn)
if duplicates.empty:
    print('No duplicate Order_ID + Product_ID combinations found.')
else:
    print(duplicates)

=== Checking for Duplicate Order + Product Combinations ===
         Order_ID       Product_ID  occurrence_count
0  CA-2015-103135  OFF-BI-10000069                 2
1  CA-2016-129714  OFF-PA-10001970                 2
2  CA-2016-137043  FUR-FU-10003664                 2
3  CA-2016-140571  OFF-PA-10001954                 2
4  CA-2017-118017  TEC-AC-10002006                 2
5  CA-2017-152912  OFF-ST-10003208                 2
6  US-2014-150119  FUR-CH-10002965                 2
7  US-2016-123750  TEC-AC-10004659                 2


In [26]:
# 6.7 Region + Category Cross Analysis
print('=== Region x Category Sales Matrix ===')
cross = pd.read_sql("""
    SELECT 
        Region,
        Category,
        ROUND(SUM(Sales), 2)  AS total_sales,
        ROUND(SUM(Profit), 2) AS total_profit
    FROM superstore
    GROUP BY Region, Category
    ORDER BY Region, total_sales DESC
""", conn)
cross

=== Region x Category Sales Matrix ===


,Region,Category,total_sales,total_profit
0,Central,Technology,170416.31,33697.43
1,Central,Office Supplies,167026.42,8879.98
2,Central,Furniture,163797.16,-2871.05
3,East,Technology,264973.98,47462.04
4,East,Furniture,208291.20,3046.17
5,East,Office Supplies,205516.05,41014.58
6,South,Technology,148771.91,19991.83
7,South,Office Supplies,125651.31,19986.39
8,South,Furniture,117298.68,6771.21
9,West,Furniture,252612.74,11504.95


---
## Step 7: Data Validation & Quality Checks

In [27]:
# 7.1 Total row count validation
row_count = pd.read_sql("SELECT COUNT(*) AS total FROM superstore", conn)
print('Total Rows:', row_count['total'][0])

Total Rows: 9994


In [28]:
# 7.2 Check for NULL values in critical columns
print('=== NULL Check on Key Columns ===')
nulls = pd.read_sql("""
    SELECT 
        SUM(CASE WHEN Order_ID     IS NULL THEN 1 ELSE 0 END) AS null_order_id,
        SUM(CASE WHEN Customer_ID  IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
        SUM(CASE WHEN Sales        IS NULL THEN 1 ELSE 0 END) AS null_sales,
        SUM(CASE WHEN Profit       IS NULL THEN 1 ELSE 0 END) AS null_profit,
        SUM(CASE WHEN Product_Name IS NULL THEN 1 ELSE 0 END) AS null_product_name
    FROM superstore
""", conn)
nulls

=== NULL Check on Key Columns ===


,null_order_id,null_customer_id,null_sales,null_profit,null_product_name
0,0,0,0,0,0


In [29]:
# 7.3 Negative profit count (loss-making orders)
loss = pd.read_sql("""
    SELECT COUNT(*) AS loss_making_orders
    FROM superstore
    WHERE Profit < 0
""", conn)
print('Loss-making orders:', loss['loss_making_orders'][0])

Loss-making orders: 1871


In [30]:
# 7.4 Sales range validation
stats = pd.read_sql("""
    SELECT 
        ROUND(MIN(Sales), 2)  AS min_sales,
        ROUND(MAX(Sales), 2)  AS max_sales,
        ROUND(AVG(Sales), 2)  AS avg_sales,
        ROUND(MIN(Profit), 2) AS min_profit,
        ROUND(MAX(Profit), 2) AS max_profit
    FROM superstore
""", conn)
stats

,min_sales,max_sales,avg_sales,min_profit,max_profit
0,0.44,22638.48,229.86,-6599.98,8399.98


In [31]:
# 7.5 Date range of dataset
dates = pd.read_sql("""
    SELECT 
        MIN(Order_Date) AS earliest_order,
        MAX(Order_Date) AS latest_order
    FROM superstore
""", conn)
dates

,earliest_order,latest_order
0,2014-01-03 00:00:00,2017-12-30 00:00:00
